# Module 04 — ML Theory & Evaluation
## 04-01: Evaluation Metrics Deep Dive

**Objective:** Implement the full suite of classification and regression evaluation metrics from scratch, understand when each metric is appropriate, and build reusable evaluator classes that validate against sklearn equivalents.

**Prerequisites:** 2-01 through 2-10 (Supervised Learning)

## Part 0 — Setup & Prerequisites

This notebook owns **precision, recall, F1 (micro/macro/weighted), ROC-AUC, precision-recall curves, confusion matrix analysis, regression metrics (MSE, MAE, R², MAPE), and calibration curves**. These are the vocabulary of model evaluation used throughout every subsequent module. We apply them to the sklearn Digits dataset (10-class classification) and the California Housing dataset (regression).

**Prerequisites:** 2-01 through 2-10 (Supervised Learning)

> **Note:** The focus here is evaluation methodology. We train small, quick models only to generate predictions for evaluation — model training is not the topic.

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import sys
import warnings
import random
import time
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap

from sklearn.datasets import load_digits, fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.ensemble import RandomForestClassifier
from sklearn import metrics as skmetrics

import torch

print(f'Python : {sys.version.split()[0]}')
print(f'NumPy  : {np.__version__}')
print(f'PyTorch: {torch.__version__}')

In [ ]:
# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 1103
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ── Constants ─────────────────────────────────────────────────────────────────
TEST_SIZE: float = 0.20          # 80/20 split
N_CALIB_BINS: int = 10           # Bins for calibration curve / ECE
N_THRESH: int = 500              # Number of thresholds for ROC / PR sweep
LR_MAX_ITER: int = 1000          # LogisticRegression solver iterations
RF_N_ESTIMATORS: int = 100       # Random Forest size

print(f'SEED       : {SEED}')
print(f'Test split : {TEST_SIZE:.0%}')
print(f'Calib bins : {N_CALIB_BINS}')

### Data Loading & Exploration

We load two datasets:
- **Digits** — 1 797 8×8 greyscale images of handwritten digits (0–9). Ten classes, relatively balanced.
- **California Housing** — 20 640 census block groups. Continuous target: median house value (regression).

In [ ]:
# ── Load Digits (classification) ──────────────────────────────────────────────
digits = load_digits()
X_digits: np.ndarray = digits.data          # (1797, 64)
y_digits: np.ndarray = digits.target        # (1797,)  values 0..9
class_names: list = [str(c) for c in digits.target_names]
n_classes: int = len(class_names)

X_tr_dig, X_te_dig, y_tr_dig, y_te_dig = train_test_split(
    X_digits, y_digits,
    test_size=TEST_SIZE,
    random_state=SEED,
    stratify=y_digits,
)
print(f'Digits — train: {X_tr_dig.shape}, test: {X_te_dig.shape}')
unique_cls, counts = np.unique(y_tr_dig, return_counts=True)
print(f'Class distribution (train): {dict(zip(unique_cls.tolist(), counts.tolist()))}')

# ── Load California Housing (regression) ──────────────────────────────────────
housing = fetch_california_housing()
X_housing: np.ndarray = housing.data        # (20640, 8)
y_housing: np.ndarray = housing.target      # (20640,) median value in $100k
feature_names_housing: list = list(housing.feature_names)

X_tr_hs, X_te_hs, y_tr_hs, y_te_hs = train_test_split(
    X_housing, y_housing,
    test_size=TEST_SIZE,
    random_state=SEED,
)
print(f'Housing — train: {X_tr_hs.shape}, test: {X_te_hs.shape}')
print(f'Target range: [{y_housing.min():.2f}, {y_housing.max():.2f}]  (median house value ×$100k)')

# ── Quick EDA ─────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 3))

ax = axes[0]
ax.bar(unique_cls, counts, color='steelblue', edgecolor='white')
ax.set_xlabel('Digit class')
ax.set_ylabel('Count')
ax.set_title('Digits: Class Distribution (Train)')
ax.set_xticks(unique_cls)

ax = axes[1]
ax.hist(y_housing, bins=50, color='salmon', edgecolor='white')
ax.set_xlabel('Median house value (×$100k)')
ax.set_ylabel('Count')
ax.set_title('California Housing: Target Distribution')

plt.tight_layout()
plt.savefig('04_01_eda.png', dpi=100, bbox_inches='tight')
plt.show()

---
## Part 1 — Evaluation Metrics from Scratch

We implement every metric from first principles using only NumPy. The logic mirrors the sklearn implementations so we can validate against them in Part 4.

### 1.1 Confusion Matrix

The confusion matrix $\mathbf{C} \in \mathbb{Z}^{K \times K}$ where $C_{ij}$ is the number of samples with true class $i$ predicted as class $j$. It is the foundation for all classification metrics.

In [ ]:
def confusion_matrix_scratch(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    n_classes: int | None = None,
) -> np.ndarray:
    '''Compute the confusion matrix from true and predicted labels.

    Args:
        y_true: 1-D array of ground-truth integer labels.
        y_pred: 1-D array of predicted integer labels, same length as y_true.
        n_classes: Number of classes. If None, inferred as max(y_true)+1.

    Returns:
        Integer array of shape (n_classes, n_classes). Entry [i, j] counts
        samples with true label i predicted as j.
    '''
    if n_classes is None:
        n_classes = int(max(y_true.max(), y_pred.max())) + 1
    cm: np.ndarray = np.zeros((n_classes, n_classes), dtype=np.int64)
    for true_label, pred_label in zip(y_true, y_pred):
        cm[int(true_label), int(pred_label)] += 1
    return cm


def plot_confusion_matrix(
    cm: np.ndarray,
    class_names: list,
    title: str = 'Confusion Matrix',
    normalise: bool = True,
    ax: plt.Axes | None = None,
) -> plt.Axes:
    '''Visualise a confusion matrix as an annotated heatmap.

    Args:
        cm: Square integer or float confusion matrix.
        class_names: List of class label strings.
        title: Plot title.
        normalise: If True, normalise each row to sum to 1 (recall per class).
        ax: Matplotlib Axes to draw on. If None, a new figure is created.

    Returns:
        The Axes object with the plot drawn.
    '''
    if normalise:
        row_sums = cm.sum(axis=1, keepdims=True)
        display_cm = np.where(row_sums > 0, cm / row_sums, 0.0)
        fmt = '.2f'
    else:
        display_cm = cm.astype(float)
        fmt = 'd'

    if ax is None:
        _, ax = plt.subplots(figsize=(8, 6))

    im = ax.imshow(display_cm, interpolation='nearest', cmap='Blues', vmin=0)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    tick_marks = np.arange(len(class_names))
    ax.set_xticks(tick_marks)
    ax.set_xticklabels(class_names, fontsize=9)
    ax.set_yticks(tick_marks)
    ax.set_yticklabels(class_names, fontsize=9)

    thresh = display_cm.max() / 2.0
    for i in range(display_cm.shape[0]):
        for j in range(display_cm.shape[1]):
            val = display_cm[i, j]
            text = f'{val:{fmt}}' if fmt == '.2f' else f'{int(cm[i, j])}'
            ax.text(j, i, text, ha='center', va='center', fontsize=7,
                    color='white' if val > thresh else 'black')

    ax.set_ylabel('True label')
    ax.set_xlabel('Predicted label')
    ax.set_title(title)
    return ax


# Quick sanity check
y_true_toy: np.ndarray = np.array([0, 0, 1, 1, 2, 2])
y_pred_toy: np.ndarray = np.array([0, 1, 1, 2, 2, 2])
cm_toy: np.ndarray = confusion_matrix_scratch(y_true_toy, y_pred_toy)
print('Toy confusion matrix (3 classes):')
print(cm_toy)
assert np.array_equal(cm_toy, skmetrics.confusion_matrix(y_true_toy, y_pred_toy)), \
    'confusion_matrix_scratch does not match sklearn!'
print('Matches sklearn ✓')

### 1.2 Precision, Recall, and F1 Score

For a single class $k$, let $TP_k$, $FP_k$, $FN_k$ be true positives, false positives, and false negatives:

$$\text{Precision}_k = \frac{TP_k}{TP_k + FP_k} \qquad \text{Recall}_k = \frac{TP_k}{TP_k + FN_k} \qquad F1_k = 2 \cdot \frac{P_k \cdot R_k}{P_k + R_k}$$

Averaging strategies:
- **Macro:** unweighted mean over classes — treats all classes equally regardless of size
- **Weighted:** mean weighted by class support — appropriate for imbalanced datasets
- **Micro:** pool all TP/FP/FN before computing — equivalent to accuracy for multi-class

In [ ]:
def _per_class_prf(
    cm: np.ndarray,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    '''Compute per-class precision, recall, F1, and support from confusion matrix.

    Args:
        cm: Square confusion matrix of shape (K, K).

    Returns:
        Tuple (precision, recall, f1, support), each a 1-D array of length K.
    '''
    tp: np.ndarray = np.diag(cm).astype(float)
    fp: np.ndarray = cm.sum(axis=0) - tp        # column sum minus diagonal
    fn: np.ndarray = cm.sum(axis=1) - tp        # row sum minus diagonal
    support: np.ndarray = cm.sum(axis=1).astype(float)

    precision: np.ndarray = np.where(tp + fp > 0, tp / (tp + fp), 0.0)
    recall: np.ndarray = np.where(tp + fn > 0, tp / (tp + fn), 0.0)
    f1: np.ndarray = np.where(
        precision + recall > 0,
        2 * precision * recall / (precision + recall),
        0.0,
    )
    return precision, recall, f1, support


def precision_score_scratch(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    average: str = 'macro',
    n_classes: int | None = None,
) -> float:
    '''Compute precision with the specified averaging strategy.

    Args:
        y_true: Ground-truth labels.
        y_pred: Predicted labels.
        average: One of 'macro', 'weighted', or 'micro'.
        n_classes: Number of classes (inferred if None).

    Returns:
        Scalar precision value.
    '''
    cm = confusion_matrix_scratch(y_true, y_pred, n_classes)
    precision, _, _, support = _per_class_prf(cm)
    if average == 'macro':
        return float(precision.mean())
    if average == 'weighted':
        return float(np.average(precision, weights=support))
    if average == 'micro':
        tp_total = float(np.diag(cm).sum())
        fp_total = float((cm.sum(axis=0) - np.diag(cm)).sum())
        return tp_total / (tp_total + fp_total) if (tp_total + fp_total) > 0 else 0.0
    raise ValueError(f'Unknown average: {average!r}')


def recall_score_scratch(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    average: str = 'macro',
    n_classes: int | None = None,
) -> float:
    '''Compute recall with the specified averaging strategy.

    Args:
        y_true: Ground-truth labels.
        y_pred: Predicted labels.
        average: One of 'macro', 'weighted', or 'micro'.
        n_classes: Number of classes (inferred if None).

    Returns:
        Scalar recall value.
    '''
    cm = confusion_matrix_scratch(y_true, y_pred, n_classes)
    _, recall, _, support = _per_class_prf(cm)
    if average == 'macro':
        return float(recall.mean())
    if average == 'weighted':
        return float(np.average(recall, weights=support))
    if average == 'micro':
        tp_total = float(np.diag(cm).sum())
        fn_total = float((cm.sum(axis=1) - np.diag(cm)).sum())
        return tp_total / (tp_total + fn_total) if (tp_total + fn_total) > 0 else 0.0
    raise ValueError(f'Unknown average: {average!r}')


def f1_score_scratch(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    average: str = 'macro',
    n_classes: int | None = None,
) -> float:
    '''Compute F1 score with the specified averaging strategy.

    Args:
        y_true: Ground-truth labels.
        y_pred: Predicted labels.
        average: One of 'macro', 'weighted', or 'micro'.
        n_classes: Number of classes (inferred if None).

    Returns:
        Scalar F1 value.
    '''
    cm = confusion_matrix_scratch(y_true, y_pred, n_classes)
    _, _, f1, support = _per_class_prf(cm)
    if average == 'macro':
        return float(f1.mean())
    if average == 'weighted':
        return float(np.average(f1, weights=support))
    if average == 'micro':
        p_micro = precision_score_scratch(y_true, y_pred, 'micro', n_classes)
        r_micro = recall_score_scratch(y_true, y_pred, 'micro', n_classes)
        denom = p_micro + r_micro
        return 2 * p_micro * r_micro / denom if denom > 0 else 0.0
    raise ValueError(f'Unknown average: {average!r}')


def classification_report_scratch(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    class_names: list | None = None,
) -> pd.DataFrame:
    '''Build a per-class precision/recall/F1/support table.

    Args:
        y_true: Ground-truth labels.
        y_pred: Predicted labels.
        class_names: Optional list of class name strings.

    Returns:
        DataFrame with columns [Precision, Recall, F1, Support] indexed by class.
    '''
    k = int(max(y_true.max(), y_pred.max())) + 1
    cm = confusion_matrix_scratch(y_true, y_pred, k)
    precision, recall, f1, support = _per_class_prf(cm)
    index = class_names if class_names is not None else [str(i) for i in range(k)]
    df = pd.DataFrame({
        'Precision': precision,
        'Recall': recall,
        'F1': f1,
        'Support': support.astype(int),
    }, index=index)
    return df


print('Per-class PRF functions defined.')
print('Testing on toy example...')
for avg in ['macro', 'weighted', 'micro']:
    p_ours = precision_score_scratch(y_true_toy, y_pred_toy, avg)
    p_sk   = skmetrics.precision_score(y_true_toy, y_pred_toy, average=avg, zero_division=0)
    assert abs(p_ours - p_sk) < 1e-9, f'Precision mismatch ({avg}): {p_ours} vs {p_sk}'

    r_ours = recall_score_scratch(y_true_toy, y_pred_toy, avg)
    r_sk   = skmetrics.recall_score(y_true_toy, y_pred_toy, average=avg, zero_division=0)
    assert abs(r_ours - r_sk) < 1e-9, f'Recall mismatch ({avg}): {r_ours} vs {r_sk}'

    f_ours = f1_score_scratch(y_true_toy, y_pred_toy, avg)
    f_sk   = skmetrics.f1_score(y_true_toy, y_pred_toy, average=avg, zero_division=0)
    assert abs(f_ours - f_sk) < 1e-9, f'F1 mismatch ({avg}): {f_ours} vs {f_sk}'
print('All averaging modes match sklearn ✓')

### 1.3 ROC Curve & AUC

The ROC curve plots **True Positive Rate** (recall) vs **False Positive Rate** as the decision threshold varies from 1 → 0:

$$\text{TPR}(t) = \frac{TP(t)}{TP(t) + FN(t)} \qquad \text{FPR}(t) = \frac{FP(t)}{FP(t) + TN(t)}$$

AUC = Area Under the ROC Curve, computed via the trapezoidal rule. AUC = 0.5 is random chance; AUC = 1.0 is perfect ranking.

For multi-class problems we compute the **One-vs-Rest** (OvR) ROC curve per class and average.

In [ ]:
def roc_curve_scratch(
    y_true_binary: np.ndarray,
    y_score: np.ndarray,
    n_thresholds: int = 500,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    '''Compute ROC curve for a binary classification problem.

    Args:
        y_true_binary: Binary ground-truth labels (0 or 1).
        y_score: Predicted probability or score for the positive class.
        n_thresholds: Number of evenly-spaced thresholds to evaluate.

    Returns:
        Tuple (fpr, tpr, thresholds) each of shape (n_thresholds + 2,), including
        sentinel points at (0, 0) and (1, 1).
    '''
    thresholds: np.ndarray = np.linspace(1.0, 0.0, n_thresholds)
    n_pos: int = int(y_true_binary.sum())
    n_neg: int = len(y_true_binary) - n_pos

    fpr_list: list = []
    tpr_list: list = []
    for t in thresholds:
        y_pred_t: np.ndarray = (y_score >= t).astype(int)
        tp = int(((y_pred_t == 1) & (y_true_binary == 1)).sum())
        fp = int(((y_pred_t == 1) & (y_true_binary == 0)).sum())
        tpr_list.append(tp / n_pos if n_pos > 0 else 0.0)
        fpr_list.append(fp / n_neg if n_neg > 0 else 0.0)

    fpr: np.ndarray = np.array([0.0] + fpr_list + [1.0])
    tpr: np.ndarray = np.array([0.0] + tpr_list + [1.0])
    thresh_out: np.ndarray = np.array([np.inf] + list(thresholds) + [0.0])
    return fpr, tpr, thresh_out


def auc_trapezoidal(x: np.ndarray, y: np.ndarray) -> float:
    '''Compute area under the curve using the trapezoidal rule.

    Args:
        x: x-coordinates, not necessarily monotone.
        y: y-coordinates of the same length as x.

    Returns:
        Scalar AUC value.
    '''
    order: np.ndarray = np.argsort(x)
    x_sorted: np.ndarray = x[order]
    y_sorted: np.ndarray = y[order]
    return float(np.trapz(y_sorted, x_sorted))


def roc_auc_ovr(
    y_true: np.ndarray,
    y_proba: np.ndarray,
    average: str = 'macro',
) -> float:
    '''Compute One-vs-Rest ROC-AUC for multi-class classification.

    Args:
        y_true: Integer class labels of shape (n_samples,).
        y_proba: Probability matrix of shape (n_samples, n_classes).
        average: 'macro' for unweighted mean, 'weighted' for support-weighted mean.

    Returns:
        Scalar averaged ROC-AUC.
    '''
    n_cls: int = y_proba.shape[1]
    aucs: list = []
    supports: list = []
    for k in range(n_cls):
        y_bin: np.ndarray = (y_true == k).astype(int)
        scores_k: np.ndarray = y_proba[:, k]
        fpr_k, tpr_k, _ = roc_curve_scratch(y_bin, scores_k, n_thresholds=N_THRESH)
        aucs.append(auc_trapezoidal(fpr_k, tpr_k))
        supports.append(int(y_bin.sum()))
    aucs_arr: np.ndarray = np.array(aucs)
    if average == 'macro':
        return float(aucs_arr.mean())
    if average == 'weighted':
        return float(np.average(aucs_arr, weights=np.array(supports, dtype=float)))
    raise ValueError(f'Unknown average: {average!r}')


# Validate on toy binary data
rng_toy = np.random.RandomState(42)
y_bin_toy: np.ndarray = rng_toy.randint(0, 2, size=200)
scores_toy: np.ndarray = rng_toy.rand(200)
fpr_t, tpr_t, _ = roc_curve_scratch(y_bin_toy, scores_toy)
auc_ours: float = auc_trapezoidal(fpr_t, tpr_t)
auc_sk: float = skmetrics.roc_auc_score(y_bin_toy, scores_toy)
print(f'ROC-AUC scratch: {auc_ours:.6f}  sklearn: {auc_sk:.6f}  diff: {abs(auc_ours - auc_sk):.2e}')
assert abs(auc_ours - auc_sk) < 0.005, 'ROC-AUC mismatch exceeds tolerance'
print('ROC curve and AUC match sklearn ✓')

### 1.4 Precision-Recall Curve & Average Precision

The PR curve plots **Precision** vs **Recall** as the threshold varies. Unlike ROC, it is sensitive to class imbalance — a random classifier traces a horizontal line at $P = \text{prevalence}$, not a diagonal. **Average Precision (AP)** is the area under the PR curve:

$$\text{AP} = \sum_n (R_n - R_{n-1}) \cdot P_n$$

Use the PR curve over ROC when: positive examples are rare (imbalanced data), or you care more about false positives than false negatives.

In [ ]:
def pr_curve_scratch(
    y_true_binary: np.ndarray,
    y_score: np.ndarray,
    n_thresholds: int = 500,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    '''Compute precision-recall curve for binary classification.

    Args:
        y_true_binary: Binary ground-truth labels (0 or 1).
        y_score: Predicted scores for the positive class.
        n_thresholds: Number of threshold values to sweep.

    Returns:
        Tuple (precision, recall, thresholds) each of length n_thresholds + 1.
        Starts at (P=1, R=0) for threshold=1 and moves toward (P=prevalence, R=1).
    '''
    thresholds: np.ndarray = np.linspace(1.0, 0.0, n_thresholds)
    n_pos: int = int(y_true_binary.sum())

    precision_list: list = [1.0]
    recall_list: list = [0.0]
    thresh_list: list = [1.0]

    for t in thresholds[1:]:
        y_pred_t: np.ndarray = (y_score >= t).astype(int)
        tp = int(((y_pred_t == 1) & (y_true_binary == 1)).sum())
        fp = int(((y_pred_t == 1) & (y_true_binary == 0)).sum())
        prec = tp / (tp + fp) if (tp + fp) > 0 else 1.0
        rec = tp / n_pos if n_pos > 0 else 0.0
        precision_list.append(prec)
        recall_list.append(rec)
        thresh_list.append(float(t))

    return (
        np.array(precision_list),
        np.array(recall_list),
        np.array(thresh_list),
    )


def average_precision_scratch(
    precision: np.ndarray,
    recall: np.ndarray,
) -> float:
    '''Compute average precision as the area under the PR curve.

    Uses the step-function approximation: AP = sum_n (R_n - R_{n-1}) * P_n.

    Args:
        precision: Precision values from pr_curve_scratch.
        recall: Recall values from pr_curve_scratch.

    Returns:
        Scalar average precision value in [0, 1].
    '''
    # Sort by increasing recall
    order: np.ndarray = np.argsort(recall)
    r_sorted: np.ndarray = recall[order]
    p_sorted: np.ndarray = precision[order]
    delta_r: np.ndarray = np.diff(r_sorted, prepend=0.0)
    return float((delta_r * p_sorted).sum())


def pr_auc_ovr(
    y_true: np.ndarray,
    y_proba: np.ndarray,
    average: str = 'macro',
) -> float:
    '''Compute One-vs-Rest average precision (PR-AUC) for multi-class.

    Args:
        y_true: Integer class labels of shape (n_samples,).
        y_proba: Probability matrix of shape (n_samples, n_classes).
        average: 'macro' or 'weighted'.

    Returns:
        Scalar averaged PR-AUC.
    '''
    n_cls: int = y_proba.shape[1]
    aps: list = []
    supports: list = []
    for k in range(n_cls):
        y_bin: np.ndarray = (y_true == k).astype(int)
        p_k, r_k, _ = pr_curve_scratch(y_bin, y_proba[:, k], n_thresholds=N_THRESH)
        aps.append(average_precision_scratch(p_k, r_k))
        supports.append(int(y_bin.sum()))
    aps_arr: np.ndarray = np.array(aps)
    if average == 'macro':
        return float(aps_arr.mean())
    if average == 'weighted':
        return float(np.average(aps_arr, weights=np.array(supports, dtype=float)))
    raise ValueError(f'Unknown average: {average!r}')


# Validate
p_t, r_t, _ = pr_curve_scratch(y_bin_toy, scores_toy, n_thresholds=N_THRESH)
ap_ours: float = average_precision_scratch(p_t, r_t)
ap_sk: float = skmetrics.average_precision_score(y_bin_toy, scores_toy)
print(f'Avg Precision scratch: {ap_ours:.6f}  sklearn: {ap_sk:.6f}  diff: {abs(ap_ours - ap_sk):.2e}')
assert abs(ap_ours - ap_sk) < 0.01, 'AP mismatch exceeds tolerance'
print('PR curve and Average Precision match sklearn ✓')

### 1.5 Calibration Curves & Expected Calibration Error (ECE)

A model is **calibrated** when its predicted probability equals the true frequency of the positive class. We measure this with a reliability diagram and the ECE:

$$\text{ECE} = \sum_{b=1}^{B} \frac{|\mathcal{B}_b|}{n} \left| \text{acc}(\mathcal{B}_b) - \text{conf}(\mathcal{B}_b) \right|$$

where $\mathcal{B}_b$ is bin $b$, $\text{acc}$ is empirical accuracy in that bin, and $\text{conf}$ is mean predicted confidence. ECE = 0 means perfect calibration.

In [ ]:
def calibration_curve_scratch(
    y_true_binary: np.ndarray,
    y_prob: np.ndarray,
    n_bins: int = 10,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    '''Compute reliability diagram data for binary calibration analysis.

    Args:
        y_true_binary: Binary ground-truth labels.
        y_prob: Predicted probability for the positive class.
        n_bins: Number of equal-width bins in [0, 1].

    Returns:
        Tuple (mean_predicted_prob, fraction_of_positives, bin_sizes). Each
        array has length n_bins; empty bins are represented by NaN.
    '''
    bin_edges: np.ndarray = np.linspace(0.0, 1.0, n_bins + 1)
    mean_prob: list = []
    frac_pos: list = []
    bin_sizes: list = []

    for lo, hi in zip(bin_edges[:-1], bin_edges[1:]):
        if hi < 1.0:
            mask: np.ndarray = (y_prob >= lo) & (y_prob < hi)
        else:
            mask = (y_prob >= lo) & (y_prob <= hi)
        count: int = int(mask.sum())
        if count == 0:
            mean_prob.append(float('nan'))
            frac_pos.append(float('nan'))
        else:
            mean_prob.append(float(y_prob[mask].mean()))
            frac_pos.append(float(y_true_binary[mask].mean()))
        bin_sizes.append(count)

    return np.array(mean_prob), np.array(frac_pos), np.array(bin_sizes)


def expected_calibration_error(
    y_true_binary: np.ndarray,
    y_prob: np.ndarray,
    n_bins: int = 10,
) -> float:
    '''Compute Expected Calibration Error (ECE) for a binary classifier.

    Args:
        y_true_binary: Binary ground-truth labels.
        y_prob: Predicted probability for the positive class.
        n_bins: Number of equal-width confidence bins.

    Returns:
        Scalar ECE in [0, 1]. Lower is better (0 = perfect calibration).
    '''
    n: int = len(y_true_binary)
    bin_edges: np.ndarray = np.linspace(0.0, 1.0, n_bins + 1)
    ece: float = 0.0

    for lo, hi in zip(bin_edges[:-1], bin_edges[1:]):
        if hi < 1.0:
            mask: np.ndarray = (y_prob >= lo) & (y_prob < hi)
        else:
            mask = (y_prob >= lo) & (y_prob <= hi)
        count: int = int(mask.sum())
        if count == 0:
            continue
        acc_b: float = float(y_true_binary[mask].mean())
        conf_b: float = float(y_prob[mask].mean())
        ece += (count / n) * abs(acc_b - conf_b)

    return ece

print('Calibration curve and ECE functions defined.')

### 1.6 Regression Metrics

For regression we measure error in the scale of the target:

| Metric | Formula | Sensitivity |
|--------|---------|-------------|
| MSE | $\frac{1}{n}\sum(y_i - \hat{y}_i)^2$ | Outliers (squared) |
| RMSE | $\sqrt{\text{MSE}}$ | Same units as target |
| MAE | $\frac{1}{n}\sum|y_i - \hat{y}_i|$ | Robust to outliers |
| $R^2$ | $1 - \frac{\text{SS}_{\text{res}}}{\text{SS}_{\text{tot}}}$ | Proportion of variance explained |
| MAPE | $\frac{100}{n}\sum\left|\frac{y_i - \hat{y}_i}{y_i}\right|$ | Relative error (%, unbounded) |

In [ ]:
def mse_scratch(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    '''Mean squared error.

    Args:
        y_true: Ground-truth target values.
        y_pred: Predicted values.

    Returns:
        Scalar MSE.
    '''
    return float(np.mean((y_true - y_pred) ** 2))


def rmse_scratch(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    '''Root mean squared error.

    Args:
        y_true: Ground-truth target values.
        y_pred: Predicted values.

    Returns:
        Scalar RMSE.
    '''
    return float(np.sqrt(mse_scratch(y_true, y_pred)))


def mae_scratch(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    '''Mean absolute error.

    Args:
        y_true: Ground-truth target values.
        y_pred: Predicted values.

    Returns:
        Scalar MAE.
    '''
    return float(np.mean(np.abs(y_true - y_pred)))


def r2_scratch(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    '''Coefficient of determination R².

    Args:
        y_true: Ground-truth target values.
        y_pred: Predicted values.

    Returns:
        Scalar R² (1.0 = perfect; 0.0 = constant baseline; can be negative).
    '''
    ss_res: float = float(np.sum((y_true - y_pred) ** 2))
    ss_tot: float = float(np.sum((y_true - y_true.mean()) ** 2))
    return 1.0 - ss_res / ss_tot if ss_tot > 0 else 0.0


def mape_scratch(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    eps: float = 1e-8,
) -> float:
    '''Mean absolute percentage error.

    Args:
        y_true: Ground-truth target values (must be non-zero).
        y_pred: Predicted values.
        eps: Small value added to denominator to prevent division by zero.

    Returns:
        Scalar MAPE as a percentage (e.g., 12.5 means 12.5%).
    '''
    return float(100.0 * np.mean(np.abs((y_true - y_pred) / (np.abs(y_true) + eps))))


# Validate against sklearn
rng_reg = np.random.RandomState(7)
y_true_reg: np.ndarray = rng_reg.randn(200) * 3 + 5
y_pred_reg: np.ndarray = y_true_reg + rng_reg.randn(200) * 1.2

metrics_check = [
    ('MSE',  mse_scratch(y_true_reg, y_pred_reg),  skmetrics.mean_squared_error(y_true_reg, y_pred_reg)),
    ('MAE',  mae_scratch(y_true_reg, y_pred_reg),  skmetrics.mean_absolute_error(y_true_reg, y_pred_reg)),
    ('R²',   r2_scratch(y_true_reg, y_pred_reg),   skmetrics.r2_score(y_true_reg, y_pred_reg)),
]
print(f'{"Metric":<6} {"Ours":>10} {"Sklearn":>10} {"Diff":>12}')
for name, ours, sk in metrics_check:
    diff = abs(ours - sk)
    print(f'{name:<6} {ours:>10.6f} {sk:>10.6f} {diff:>12.2e}')
    assert diff < 1e-9, f'{name} mismatch!'
print('All regression metrics match sklearn ✓')

---
## Part 2 — Assembling the Evaluation Harnesses

We wrap the Part 1 primitives into two reusable classes:
- `ClassificationEvaluator` — accepts predictions and probabilities, computes all classification metrics
- `RegressionEvaluator` — accepts predictions, computes all regression metrics

Both classes produce a summary DataFrame for easy comparison across models.

In [ ]:
class ClassificationEvaluator:
    '''Comprehensive evaluator for multi-class classification models.

    Implements precision, recall, F1 (macro/weighted/micro), confusion matrix,
    ROC-AUC (OvR), PR-AUC (OvR), and calibration metrics from scratch.

    Attributes:
        n_classes: Number of target classes.
        class_names: List of class label strings.
        n_calib_bins: Number of bins for ECE computation.
    '''

    def __init__(
        self,
        n_classes: int,
        class_names: list | None = None,
        n_calib_bins: int = 10,
    ) -> None:
        '''Initialise ClassificationEvaluator.

        Args:
            n_classes: Number of distinct target classes.
            class_names: Optional list of class label strings (length n_classes).
            n_calib_bins: Bins for the ECE / calibration curve.
        '''
        self.n_classes: int = n_classes
        self.class_names: list = (
            class_names if class_names is not None
            else [str(i) for i in range(n_classes)]
        )
        self.n_calib_bins: int = n_calib_bins

    def evaluate(
        self,
        y_true: np.ndarray,
        y_pred: np.ndarray,
        y_proba: np.ndarray | None = None,
        model_name: str = 'Model',
    ) -> dict:
        '''Compute all classification metrics for a set of predictions.

        Args:
            y_true: Ground-truth integer class labels.
            y_pred: Predicted integer class labels.
            y_proba: Optional probability matrix (n_samples, n_classes) for
                ROC-AUC, PR-AUC, and calibration metrics.
            model_name: Label for this model in the results dict.

        Returns:
            Dictionary of metric name → scalar value.
        '''
        results: dict = {'Model': model_name}

        # Accuracy
        results['Accuracy'] = float((y_true == y_pred).mean())

        # Precision / Recall / F1
        for avg in ['macro', 'weighted', 'micro']:
            tag = avg.capitalize()
            results[f'Precision ({tag})'] = precision_score_scratch(
                y_true, y_pred, avg, self.n_classes)
            results[f'Recall ({tag})'] = recall_score_scratch(
                y_true, y_pred, avg, self.n_classes)
            results[f'F1 ({tag})'] = f1_score_scratch(
                y_true, y_pred, avg, self.n_classes)

        # Probability-based metrics
        if y_proba is not None:
            results['ROC-AUC (Macro OvR)'] = roc_auc_ovr(y_true, y_proba, 'macro')
            results['ROC-AUC (Weighted OvR)'] = roc_auc_ovr(y_true, y_proba, 'weighted')
            results['PR-AUC (Macro OvR)'] = pr_auc_ovr(y_true, y_proba, 'macro')
            results['PR-AUC (Weighted OvR)'] = pr_auc_ovr(y_true, y_proba, 'weighted')

            # ECE: average over all OvR binary problems
            eces: list = []
            for k in range(self.n_classes):
                y_bin_k = (y_true == k).astype(int)
                eces.append(expected_calibration_error(
                    y_bin_k, y_proba[:, k], self.n_calib_bins))
            results['ECE (mean OvR)'] = float(np.mean(eces))

        return results

    def per_class_report(
        self,
        y_true: np.ndarray,
        y_pred: np.ndarray,
    ) -> pd.DataFrame:
        '''Return a per-class precision/recall/F1/support DataFrame.

        Args:
            y_true: Ground-truth labels.
            y_pred: Predicted labels.

        Returns:
            DataFrame indexed by class name with columns Precision/Recall/F1/Support.
        '''
        return classification_report_scratch(y_true, y_pred, self.class_names)

    def plot_confusion(
        self,
        y_true: np.ndarray,
        y_pred: np.ndarray,
        title: str = 'Confusion Matrix',
        normalise: bool = True,
    ) -> None:
        '''Plot the normalised confusion matrix.

        Args:
            y_true: Ground-truth labels.
            y_pred: Predicted labels.
            title: Plot title string.
            normalise: Row-normalise so each entry shows recall per class.
        '''
        cm = confusion_matrix_scratch(y_true, y_pred, self.n_classes)
        fig, ax = plt.subplots(figsize=(8, 6))
        plot_confusion_matrix(cm, self.class_names, title=title,
                              normalise=normalise, ax=ax)
        plt.tight_layout()
        plt.show()


class RegressionEvaluator:
    '''Evaluator for regression models.

    Computes MSE, RMSE, MAE, R², and MAPE from scratch.

    Attributes:
        target_name: Display name for the target variable.
    '''

    def __init__(self, target_name: str = 'target') -> None:
        '''Initialise RegressionEvaluator.

        Args:
            target_name: Human-readable name for the regression target.
        '''
        self.target_name: str = target_name

    def evaluate(
        self,
        y_true: np.ndarray,
        y_pred: np.ndarray,
        model_name: str = 'Model',
    ) -> dict:
        '''Compute all regression metrics.

        Args:
            y_true: Ground-truth continuous target values.
            y_pred: Predicted values.
            model_name: Label for this model.

        Returns:
            Dictionary of metric name → scalar value.
        '''
        return {
            'Model': model_name,
            'MSE': mse_scratch(y_true, y_pred),
            'RMSE': rmse_scratch(y_true, y_pred),
            'MAE': mae_scratch(y_true, y_pred),
            'R²': r2_scratch(y_true, y_pred),
            'MAPE (%)': mape_scratch(y_true, y_pred),
        }

    def plot_residuals(
        self,
        y_true: np.ndarray,
        y_pred: np.ndarray,
        model_name: str = 'Model',
    ) -> None:
        '''Plot predicted vs actual and residual histogram.

        Args:
            y_true: Ground-truth values.
            y_pred: Predicted values.
            model_name: Label used in plot title.
        '''
        residuals: np.ndarray = y_true - y_pred
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))

        ax = axes[0]
        lims = [min(y_true.min(), y_pred.min()), max(y_true.max(), y_pred.max())]
        ax.scatter(y_true, y_pred, alpha=0.3, s=10, color='steelblue')
        ax.plot(lims, lims, 'r--', lw=2, label='Perfect fit')
        ax.set_xlabel(f'True {self.target_name}')
        ax.set_ylabel(f'Predicted {self.target_name}')
        ax.set_title(f'{model_name}: Predicted vs True')
        ax.legend()
        ax.grid(True, alpha=0.3)

        ax = axes[1]
        ax.hist(residuals, bins=40, color='salmon', edgecolor='white')
        ax.axvline(0, color='black', lw=2)
        ax.set_xlabel('Residual (True − Predicted)')
        ax.set_ylabel('Count')
        ax.set_title(f'{model_name}: Residual Distribution')
        ax.grid(True, alpha=0.3)

        plt.tight_layout()
        plt.show()


# Sanity check on toy data
clf_eval = ClassificationEvaluator(n_classes=3, class_names=['A', 'B', 'C'])
result_toy = clf_eval.evaluate(y_true_toy, y_pred_toy)
print('ClassificationEvaluator sanity check:')
print(f'  Accuracy         : {result_toy["Accuracy"]:.4f}')
print(f'  F1 (Macro)       : {result_toy["F1 (Macro)"]:.4f}')

reg_eval = RegressionEvaluator(target_name='value')
result_reg_toy = reg_eval.evaluate(y_true_reg, y_pred_reg)
print('RegressionEvaluator sanity check:')
print(f'  RMSE: {result_reg_toy["RMSE"]:.4f}  R²: {result_reg_toy["R²"]:.4f}')

---
## Part 3 — Applying to Model Outputs

We train two small, quick models — one for each dataset — and then apply our evaluators. The training is minimal; the notebook's focus is metric interpretation.

**Classifier:** Logistic Regression and Random Forest on Digits.  
**Regressor:** Ridge Regression on California Housing.

In [ ]:
# ── Train classifiers on Digits ───────────────────────────────────────────────
scaler_dig = StandardScaler()
X_tr_dig_sc: np.ndarray = scaler_dig.fit_transform(X_tr_dig)
X_te_dig_sc: np.ndarray = scaler_dig.transform(X_te_dig)

print('Training classifiers on Digits...')

t0 = time.perf_counter()
lr_model = LogisticRegression(max_iter=LR_MAX_ITER, random_state=SEED, C=1.0)
lr_model.fit(X_tr_dig_sc, y_tr_dig)
lr_time: float = time.perf_counter() - t0

t0 = time.perf_counter()
rf_model = RandomForestClassifier(
    n_estimators=RF_N_ESTIMATORS, random_state=SEED, n_jobs=-1)
rf_model.fit(X_tr_dig, y_tr_dig)   # RF does not need scaling
rf_time: float = time.perf_counter() - t0

print(f'  Logistic Regression trained in {lr_time:.2f}s')
print(f'  Random Forest       trained in {rf_time:.2f}s')

# Get predictions
y_pred_lr: np.ndarray = lr_model.predict(X_te_dig_sc)
y_proba_lr: np.ndarray = lr_model.predict_proba(X_te_dig_sc)
y_pred_rf: np.ndarray = rf_model.predict(X_te_dig)
y_proba_rf: np.ndarray = rf_model.predict_proba(X_te_dig)

# ── Train regressor on California Housing ────────────────────────────────────
scaler_hs = StandardScaler()
X_tr_hs_sc: np.ndarray = scaler_hs.fit_transform(X_tr_hs)
X_te_hs_sc: np.ndarray = scaler_hs.transform(X_te_hs)

print('\nTraining Ridge regressor on California Housing...')
t0 = time.perf_counter()
ridge_model = Ridge(alpha=1.0)
ridge_model.fit(X_tr_hs_sc, y_tr_hs)
ridge_time: float = time.perf_counter() - t0
print(f'  Ridge trained in {ridge_time:.3f}s')

y_pred_ridge: np.ndarray = ridge_model.predict(X_te_hs_sc)
print('All models trained and predictions generated.')

In [ ]:
# ── Evaluate classifiers ───────────────────────────────────────────────────────
clf_eval_digits = ClassificationEvaluator(
    n_classes=n_classes,
    class_names=class_names,
    n_calib_bins=N_CALIB_BINS,
)

results_lr: dict = clf_eval_digits.evaluate(
    y_te_dig, y_pred_lr, y_proba_lr, 'Logistic Regression')
results_rf: dict = clf_eval_digits.evaluate(
    y_te_dig, y_pred_rf, y_proba_rf, 'Random Forest')

# Summary table
summary_clf: pd.DataFrame = pd.DataFrame([results_lr, results_rf]).set_index('Model')
print('=== Classification Evaluation Summary (Digits Test Set) ===')
print(summary_clf.round(4).to_string())

# Per-class report for Logistic Regression
print('\n=== Logistic Regression — Per-Class Report ===')
report_lr: pd.DataFrame = clf_eval_digits.per_class_report(y_te_dig, y_pred_lr)
print(report_lr.round(4).to_string())

In [ ]:
# ── Evaluate regression ────────────────────────────────────────────────────────
reg_eval_hs = RegressionEvaluator(target_name='House Value (×$100k)')

# Baseline: always predict mean of training set
y_baseline: np.ndarray = np.full_like(y_te_hs, y_tr_hs.mean())
results_baseline: dict = reg_eval_hs.evaluate(y_te_hs, y_baseline, 'Mean Baseline')
results_ridge: dict = reg_eval_hs.evaluate(y_te_hs, y_pred_ridge, 'Ridge Regression')

summary_reg: pd.DataFrame = pd.DataFrame([results_baseline, results_ridge]).set_index('Model')
print('=== Regression Evaluation Summary (Housing Test Set) ===')
print(summary_reg.round(4).to_string())

# Residual plots
reg_eval_hs.plot_residuals(y_te_hs, y_pred_ridge, 'Ridge Regression')

---
## Part 4 — Analysis & Evaluation Deep Dives

We now analyse the results in depth:
1. Compare our metrics against sklearn to confirm correctness
2. Plot ROC and PR curves and interpret them
3. Plot the confusion matrix and identify the hardest digit pairs
4. Visualise calibration — are the predicted probabilities trustworthy?

In [ ]:
# ── Validate our metrics against sklearn ──────────────────────────────────────
print('=== Scratch vs sklearn Validation (Logistic Regression on Digits) ===')
print(f'{"Metric":<30} {"Scratch":>10} {"Sklearn":>10} {"Diff":>12}')
print('-' * 65)

checks: list = [
    ('Accuracy',
     results_lr['Accuracy'],
     skmetrics.accuracy_score(y_te_dig, y_pred_lr)),
    ('F1 Macro',
     results_lr['F1 (Macro)'],
     skmetrics.f1_score(y_te_dig, y_pred_lr, average='macro', zero_division=0)),
    ('F1 Weighted',
     results_lr['F1 (Weighted)'],
     skmetrics.f1_score(y_te_dig, y_pred_lr, average='weighted', zero_division=0)),
    ('F1 Micro',
     results_lr['F1 (Micro)'],
     skmetrics.f1_score(y_te_dig, y_pred_lr, average='micro', zero_division=0)),
    ('ROC-AUC (Macro OvR)',
     results_lr['ROC-AUC (Macro OvR)'],
     skmetrics.roc_auc_score(y_te_dig, y_proba_lr, multi_class='ovr', average='macro')),
]

all_close: bool = True
for name, ours, sk in checks:
    diff = abs(ours - sk)
    status = '✓' if diff < 0.01 else '✗'
    print(f'{name:<30} {ours:>10.6f} {sk:>10.6f} {diff:>12.2e} {status}')
    if diff >= 0.01:
        all_close = False

print()
print(f'All metrics within tolerance: {all_close}')

In [ ]:
# ── ROC and PR curves for selected digit classes ───────────────────────────────
selected_classes: list = [0, 1, 4, 8, 9]   # Interesting digits to highlight
palette: list = ['#e41a1c', '#377eb8', '#4daf4a', '#984ea3', '#ff7f00']

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax_roc = axes[0]
ax_pr = axes[1]

ax_roc.plot([0, 1], [0, 1], 'k--', lw=1.5, label='Random (AUC=0.5)')

for cls_idx, (cls, col) in enumerate(zip(selected_classes, palette)):
    y_bin_cls: np.ndarray = (y_te_dig == cls).astype(int)
    scores_cls: np.ndarray = y_proba_lr[:, cls]

    # ROC
    fpr_cls, tpr_cls, _ = roc_curve_scratch(y_bin_cls, scores_cls, N_THRESH)
    auc_cls: float = auc_trapezoidal(fpr_cls, tpr_cls)
    ax_roc.plot(fpr_cls, tpr_cls, color=col, lw=2,
                label=f'Digit {cls} (AUC={auc_cls:.3f})')

    # PR
    p_cls, r_cls, _ = pr_curve_scratch(y_bin_cls, scores_cls, N_THRESH)
    ap_cls: float = average_precision_scratch(p_cls, r_cls)
    ax_pr.plot(r_cls, p_cls, color=col, lw=2,
               label=f'Digit {cls} (AP={ap_cls:.3f})')

ax_roc.set_xlabel('False Positive Rate')
ax_roc.set_ylabel('True Positive Rate')
ax_roc.set_title('ROC Curves (OvR) — Logistic Regression on Digits')
ax_roc.legend(fontsize=8)
ax_roc.grid(True, alpha=0.3)

ax_pr.set_xlabel('Recall')
ax_pr.set_ylabel('Precision')
ax_pr.set_title('Precision-Recall Curves (OvR) — Logistic Regression on Digits')
ax_pr.legend(fontsize=8)
ax_pr.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('04_01_roc_pr.png', dpi=100, bbox_inches='tight')
plt.show()

print('Observations:')
print('  - Most digit classes achieve AUC > 0.99 (nearly perfect binary discrimination).')
print('  - The PR curve shows that at high-recall operating points, precision stays high.')
print('  - Digit 8 is typically the hardest to separate from others.')

In [ ]:
# ── Confusion matrix comparison: LR vs RF ─────────────────────────────────────
cm_lr: np.ndarray = confusion_matrix_scratch(y_te_dig, y_pred_lr, n_classes)
cm_rf: np.ndarray = confusion_matrix_scratch(y_te_dig, y_pred_rf, n_classes)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_confusion_matrix(cm_lr, class_names,
                      title='Logistic Regression — Confusion Matrix (Normalised)',
                      normalise=True, ax=axes[0])
plot_confusion_matrix(cm_rf, class_names,
                      title='Random Forest — Confusion Matrix (Normalised)',
                      normalise=True, ax=axes[1])
plt.tight_layout()
plt.savefig('04_01_confusion.png', dpi=100, bbox_inches='tight')
plt.show()

# Most confused pairs for LR
print('\nTop-5 most confused digit pairs (Logistic Regression):')
print(f'{"True":>6} {"Pred":>6} {"Count":>8}')
off_diag_indices: list = []
for i in range(n_classes):
    for j in range(n_classes):
        if i != j and cm_lr[i, j] > 0:
            off_diag_indices.append((cm_lr[i, j], i, j))
off_diag_indices.sort(reverse=True)
for cnt, true_cls, pred_cls in off_diag_indices[:5]:
    print(f'{class_names[true_cls]:>6} {class_names[pred_cls]:>6} {cnt:>8}')

In [ ]:
# ── Calibration analysis ───────────────────────────────────────────────────────
# Examine calibration for digit class 1 (vs all others)
cls_calib: int = 1
y_bin_calib: np.ndarray = (y_te_dig == cls_calib).astype(int)

mean_pred_lr, frac_pos_lr, sizes_lr = calibration_curve_scratch(
    y_bin_calib, y_proba_lr[:, cls_calib], N_CALIB_BINS)
mean_pred_rf, frac_pos_rf, sizes_rf = calibration_curve_scratch(
    y_bin_calib, y_proba_rf[:, cls_calib], N_CALIB_BINS)

ece_lr: float = expected_calibration_error(y_bin_calib, y_proba_lr[:, cls_calib], N_CALIB_BINS)
ece_rf: float = expected_calibration_error(y_bin_calib, y_proba_rf[:, cls_calib], N_CALIB_BINS)

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot([0, 1], [0, 1], 'k--', lw=2, label='Perfect calibration')

valid_lr: np.ndarray = ~np.isnan(mean_pred_lr)
ax.plot(mean_pred_lr[valid_lr], frac_pos_lr[valid_lr], 'bo-', lw=2, ms=8,
        label=f'Logistic Regression (ECE={ece_lr:.4f})')

valid_rf: np.ndarray = ~np.isnan(mean_pred_rf)
ax.plot(mean_pred_rf[valid_rf], frac_pos_rf[valid_rf], 'rs-', lw=2, ms=8,
        label=f'Random Forest (ECE={ece_rf:.4f})')

ax.set_xlabel('Mean Predicted Probability')
ax.set_ylabel('Fraction of Positives')
ax.set_title(f'Reliability Diagram — Digit {cls_calib} (OvR)')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('04_01_calibration.png', dpi=100, bbox_inches='tight')
plt.show()

print(f'LR ECE for digit {cls_calib}: {ece_lr:.4f} (lower = better calibrated)')
print(f'RF ECE for digit {cls_calib}: {ece_rf:.4f}')
print('Logistic Regression is typically better calibrated than Random Forest.')
print('RF tends to output overconfident probabilities near 0 and 1.')

In [ ]:
# ── Metric sensitivity to class imbalance ─────────────────────────────────────
# Create an artificially imbalanced binary version of the Digits dataset
# Positive class: digit 1 (~10% of data); Negative class: all others
print('=== Metric Sensitivity: Balanced vs Imbalanced Evaluation ===')

y_imb_true: np.ndarray = (y_te_dig == 1).astype(int)  # ~10% positive
y_imb_pred_lr: np.ndarray = (y_pred_lr == 1).astype(int)
y_imb_prob_lr: np.ndarray = y_proba_lr[:, 1]

prevalence: float = float(y_imb_true.mean())
print(f'Positive prevalence: {prevalence:.2%} ({y_imb_true.sum()} / {len(y_imb_true)})')

# Majority-class baseline (always predicts 0)
y_majority_pred: np.ndarray = np.zeros_like(y_imb_true)

# Compare metrics for LR vs Majority Baseline
comparison_data: list = []
for model_name, y_p, y_prob in [
    ('Majority Baseline', y_majority_pred, None),
    ('Logistic Regression', y_imb_pred_lr, y_imb_prob_lr),
]:
    tp = int(((y_p == 1) & (y_imb_true == 1)).sum())
    fp = int(((y_p == 1) & (y_imb_true == 0)).sum())
    fn = int(((y_p == 0) & (y_imb_true == 1)).sum())
    tn = int(((y_p == 0) & (y_imb_true == 0)).sum())

    accuracy = (tp + tn) / len(y_imb_true)
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
    roc_auc_val = auc_trapezoidal(
        *roc_curve_scratch(y_imb_true, y_prob, N_THRESH)[:2]
    ) if y_prob is not None else 0.5
    pr_ap = average_precision_scratch(
        *pr_curve_scratch(y_imb_true, y_prob, N_THRESH)[:2]
    ) if y_prob is not None else prevalence

    comparison_data.append({
        'Model': model_name,
        'Accuracy': accuracy,
        'Precision': prec,
        'Recall': rec,
        'F1': f1,
        'ROC-AUC': roc_auc_val,
        'PR-AUC': pr_ap,
    })

imb_df: pd.DataFrame = pd.DataFrame(comparison_data).set_index('Model')
print(imb_df.round(4).to_string())
print()
print('Key insight: The Majority Baseline achieves ~90% Accuracy by always predicting "not 1",')
print('  yet its F1=0, ROC-AUC=0.5, and PR-AUC=prevalence reveal it learns nothing useful.')
print('  Always evaluate imbalanced tasks with F1, PR-AUC, and ROC-AUC — not just accuracy.')

In [ ]:
# ── Threshold tuning on the PR curve ──────────────────────────────────────────
# Find the threshold that maximises F1 for digit-1 detection
print('=== Threshold Tuning via Precision-Recall Curve (Digit 1 Detection) ===')

p_arr, r_arr, t_arr = pr_curve_scratch(y_imb_true, y_imb_prob_lr, N_THRESH)
f1_arr: np.ndarray = np.where(
    p_arr + r_arr > 0,
    2 * p_arr * r_arr / (p_arr + r_arr),
    0.0,
)

best_idx: int = int(np.argmax(f1_arr))
best_threshold: float = float(t_arr[best_idx])
best_f1: float = float(f1_arr[best_idx])
best_prec: float = float(p_arr[best_idx])
best_rec: float = float(r_arr[best_idx])

print(f'Default threshold (0.5): '
      f'F1={f1_score_scratch(y_imb_true, y_imb_pred_lr):.4f}')
print(f'Optimal threshold ({best_threshold:.3f}): '
      f'F1={best_f1:.4f}  P={best_prec:.4f}  R={best_rec:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
ax.plot(r_arr, p_arr, 'b-', lw=2, label='PR curve')
ax.scatter(best_rec, best_prec, s=120, c='red', zorder=5,
           label=f'Best F1 @ t={best_threshold:.3f}')
ax.axhline(prevalence, color='grey', ls='--', label=f'Random baseline (P={prevalence:.2f})')
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('PR Curve with Optimal F1 Threshold')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

ax = axes[1]
# Only plot thresholds where t_arr is finite
finite_mask: np.ndarray = np.isfinite(t_arr)
ax.plot(t_arr[finite_mask], p_arr[finite_mask], 'b-', lw=2, label='Precision')
ax.plot(t_arr[finite_mask], r_arr[finite_mask], 'g-', lw=2, label='Recall')
ax.plot(t_arr[finite_mask], f1_arr[finite_mask], 'r-', lw=2, label='F1')
ax.axvline(best_threshold, color='black', ls='--', lw=1.5,
           label=f'Best threshold={best_threshold:.3f}')
ax.set_xlabel('Decision Threshold')
ax.set_ylabel('Score')
ax.set_title('P / R / F1 vs Threshold')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('04_01_threshold.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# ── Regression deep dive: error distribution and outlier sensitivity ───────────
print('=== Regression Analysis: Outlier Sensitivity of MSE vs MAE ===')

residuals_ridge: np.ndarray = y_te_hs - y_pred_ridge
abs_errors: np.ndarray = np.abs(residuals_ridge)

# Identify high-error samples (top 5% by absolute error)
error_threshold_95: float = float(np.percentile(abs_errors, 95))
high_error_mask: np.ndarray = abs_errors > error_threshold_95
n_high: int = int(high_error_mask.sum())

# Compute MSE and MAE with and without top-5% outlier predictions
mse_all = mse_scratch(y_te_hs, y_pred_ridge)
mse_clean = mse_scratch(y_te_hs[~high_error_mask], y_pred_ridge[~high_error_mask])
mae_all = mae_scratch(y_te_hs, y_pred_ridge)
mae_clean = mae_scratch(y_te_hs[~high_error_mask], y_pred_ridge[~high_error_mask])

print(f'High-error samples (top 5% by |residual|): {n_high} / {len(y_te_hs)}')
print(f'Error threshold: |residual| > {error_threshold_95:.3f} (×$100k)')
print()
print(f'{"Metric":<10} {"All samples":>14} {"Without top-5%":>16} {"Δ%":>10}')
print('-' * 54)
print(f'{"MSE":<10} {mse_all:>14.4f} {mse_clean:>16.4f} '
      f'{100*(mse_all - mse_clean)/mse_all:>9.1f}%')
print(f'{"MAE":<10} {mae_all:>14.4f} {mae_clean:>16.4f} '
      f'{100*(mae_all - mae_clean)/mae_all:>9.1f}%')
print()
print('MSE is much more sensitive to large errors (squared term amplifies outliers).')
print('MAE is more robust — prefer MAE when outliers are expected noise, not signal.')

# Plot error distribution with log scale to show heavy tail
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
ax.hist(abs_errors[~high_error_mask], bins=50, color='steelblue',
        edgecolor='white', label=f'Normal errors ({len(y_te_hs) - n_high})')
ax.hist(abs_errors[high_error_mask], bins=20, color='red',
        edgecolor='white', alpha=0.7, label=f'High errors ({n_high})')
ax.axvline(error_threshold_95, color='black', ls='--', lw=2,
           label=f'95th percentile={error_threshold_95:.2f}')
ax.set_xlabel('Absolute Error (×$100k)')
ax.set_ylabel('Count')
ax.set_title('Distribution of Absolute Errors')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.scatter(y_te_hs[~high_error_mask], abs_errors[~high_error_mask],
           alpha=0.2, s=8, color='steelblue', label='Normal')
ax.scatter(y_te_hs[high_error_mask], abs_errors[high_error_mask],
           alpha=0.6, s=20, color='red', label='High error')
ax.set_xlabel('True House Value (×$100k)')
ax.set_ylabel('Absolute Error')
ax.set_title('Error vs True Value (high errors in red)')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('04_01_regression_errors.png', dpi=100, bbox_inches='tight')
plt.show()

---
## Part 5 — Summary & Lessons Learned

### Key Takeaways

1. **Accuracy is a misleading metric for imbalanced datasets.** A classifier that always predicts the majority class achieves ~90% accuracy on a 10%-positive dataset, yet has F1=0 and AUC=0.5. Always report F1, ROC-AUC, and PR-AUC alongside accuracy.

2. **ROC vs Precision-Recall.** ROC curves are optimistic on imbalanced data because FPR uses the large number of true negatives in the denominator. Prefer PR curves when positive examples are rare — the baseline for a random classifier is the prevalence, not 0.5.

3. **Threshold tuning unlocks the precision-recall tradeoff.** The default 0.5 threshold is rarely optimal. Sweeping thresholds along the PR curve lets you choose the operating point that best fits the application's cost of false positives vs false negatives.

4. **MSE vs MAE: sensitivity to outliers.** MSE penalises large errors quadratically, making it sensitive to outliers. When large errors represent rare but important failures, MSE is appropriate. When they represent noise, MAE or RMSE with outlier filtering is more informative.

5. **Calibration matters for decision-making.** A model with 95% accuracy but poor calibration cannot be trusted for risk scoring. Logistic Regression is typically better calibrated than tree ensembles, which tend toward overconfident probabilities near 0 and 1.

### What's Next

→ **04-02 (Cross-Validation & Hyperparameter Tuning)** builds on these metrics to evaluate models robustly across multiple data splits and tune hyperparameters without leaking information from the test set.